In [3]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

# ─────────────────────────────────────────────
# PRETTY OUTPUT — MODULES LOADED
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📦  MODULES LOADED\033[0m")
print("\033[90m  Standard · Data · LLM · Agent libraries ready\033[0m")
print("═" * 68)

modules = [
    ("📚 Standard",  ["json", "os", "time", "pathlib", "typing"]),
    ("📊 Data",      ["pandas", "plotly.express"]),
    ("🔐 Config",    ["dotenv"]),
    ("💻 Notebook",  ["IPython.display"]),
    ("🦜 LangChain", ["langchain.agents", "langchain_core.tools", "langchain_openai"]),
    ("🤖 LLM APIs",  ["openai", "openrouter"]),
    ("✅ Validation",["pydantic"]),
]

for category, mods in modules:
    print(f"\n  \033[1;33m{category}\033[0m")
    for m in mods:
        print(f"     \033[1;32m✓\033[0m \033[90m{m}\033[0m")

total = sum(len(mods) for _, mods in modules)
print(f"\n  \033[1;34m📊 Total:\033[0m \033[1;32m{total}\033[0m modules imported successfully")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📦  MODULES LOADED
  Standard · Data · LLM · Agent libraries ready
════════════════════════════════════════════════════════════════════

  📚 Standard
     ✓ json
     ✓ os
     ✓ time
     ✓ pathlib
     ✓ typing

  📊 Data
     ✓ pandas
     ✓ plotly.express

  🔐 Config
     ✓ dotenv

  💻 Notebook
     ✓ IPython.display

  🦜 LangChain
     ✓ langchain.agents
     ✓ langchain_core.tools
     ✓ langchain_openai

  🤖 LLM APIs
     ✓ openai
     ✓ openrouter

  ✅ Validation
     ✓ pydantic

  📊 Total: 15 modules imported successfully

════════════════════════════════════════════════════════════════════



## OpenRouter

In [5]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# ─────────────────────────────────────────────
# PRETTY OUTPUT — OPENROUTER CLIENT
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  🌐  OPENROUTER CLIENT\033[0m")
print("═" * 68)

if openrouter_api_key:
    # Show masked key for security
    masked_key = f"{openrouter_api_key[:8]}...{openrouter_api_key[-4:]}" if len(openrouter_api_key) > 12 else "***"
    print(f"\n  \033[1;34m▸ API Key\033[0m      → \033[1;32m✅ FOUND\033[0m  \033[90m({masked_key})\033[0m")
    print(f"  \033[1;34m▸ Base URL\033[0m     → \033[90m{OPENROUTER_BASE_URL}\033[0m")
    print(f"  \033[1;34m▸ Timeout\033[0m      → \033[1;37m120\033[0m sec")
    print(f"  \033[1;34m▸ Max retries\033[0m  → \033[1;37m5\033[0m")
    status_line = "\033[1;32m✅ Client initialized successfully\033[0m"
else:
    print(f"\n  \033[1;34m▸ API Key\033[0m      → \033[1;31m❌ MISSING\033[0m")
    print(f"  \033[1;34m▸ Base URL\033[0m     → \033[90m{OPENROUTER_BASE_URL}\033[0m")
    print(f"\n  \033[1;33m⚠️  No OpenRouter API key found.\033[0m")
    print(f"     \033[90mAdd OPENROUTER_API_KEY to the .env file\033[0m")
    status_line = "\033[1;31m❌ Client not initialized (missing key)\033[0m"

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

print(f"\n  {status_line}")
print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🌐  OPENROUTER CLIENT
════════════════════════════════════════════════════════════════════

  ▸ API Key      → ✅ FOUND  (sk-or-v1...67a3)
  ▸ Base URL     → https://openrouter.ai/api/v1
  ▸ Timeout      → 120 sec
  ▸ Max retries  → 5

  ✅ Client initialized successfully

════════════════════════════════════════════════════════════════════



## OpenRouter

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

model_id = "openrouter/free"

print("\n" + "═" * 68)
print("\033[1;36m  🏴‍☠️  OPENROUTER — FREE MODELS ROUTER\033[0m")
print("═" * 68)
print(f"\n  \033[1;34m▸ Model\033[0m        → \033[1;37m{model_id}\033[0m  \033[90m(auto-selects free model)\033[0m")
print(f"  \033[1;34m▸ System\033[0m       → \033[90mPirate style\033[0m")
print(f"  \033[1;34m▸ Question\033[0m     → \033[90mWhat is the meaning of the universe?\033[0m")

try:
    response = openrouter.chat.completions.create(
        model=model_id,
        messages=messages
    )
    
    answer = response.choices[0].message.content
    # Which model has been selected
    actual_model = response.model
    
    print(f"\n  \033[1;32m✅ Response received\033[0m")
    print(f"  \033[1;34m▸ Routed to\033[0m    → \033[1;33m{actual_model}\033[0m")
    print(f"\n  \033[1;33m🏴‍☠️  Pirate says:\033[0m")
    print(f"     \033[90m{answer}\033[0m")
    
except Exception as e:
    print(f"\n  \033[1;31m❌ Error:\033[0m \033[90m{e}\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🏴‍☠️  OPENROUTER — FREE MODELS ROUTER
════════════════════════════════════════════════════════════════════

  ▸ Model        → openrouter/free  (auto-selects free model)
  ▸ System       → Pirate style
  ▸ Question     → What is the meaning of the universe?

  ✅ Response received
  ▸ Routed to    → nvidia/nemotron-3.5-super-vl:free

  🏴‍☠️  Pirate says:
     Ahoy there, ye curious landlubber! 

Shiver me timbers, that be a question that could make a Kraken scratch its head! Ye be askin' about the meanin' of the whole wide universe? 

Well, listen close: I reckon the universe be like a endless, glitterin' ocean, and we're all just sailors searchin' for the X that marks the spot. Some say the meanin' be gold and jewels; others say it be the thrill of the chase, the salt in the air, and the wind in yer sails.

Perhaps the true treasure ain't some chest buried on a distant shore, but the voyage itself—the storms we wea

## Data Loading

In [10]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


# ─────────────────────────────────────────────
# OUTPUT — DATASET SUMMARY
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📂  CLAIMS DATASET\033[0m")
print("\033[90m  JSON seed files · claims + handler decisions\033[0m")
print("═" * 68)

print(f"\n  \033[1;33m⚙️   Configuration:\033[0m")
print(f"     \033[90m▸ Data dir\033[0m        → \033[1;37m{DATA}\033[0m")
print(f"     \033[90m▸ Main files\033[0m      → \033[37mclaims.json\033[0m + \033[37mevals/decisions.json\033[0m")
print(f"     \033[90m▸ Held-out files\033[0m  → \033[37mevals/held_out_claims.json\033[0m + \033[37mevals/held_out_decisions.json\033[0m")

print(f"\n  \033[1;33m📋  Dataset variants:\033[0m")
print(f"     \033[1;34m[1]\033[0m \033[1;37mget_sample_data()\033[0m                  → full queue")
print(f"     \033[1;34m[2]\033[0m \033[1;37mget_sample_data(tiny=True)\033[0m         → 5-claim subset")
print(f"     \033[1;34m[3]\033[0m \033[1;37mget_sample_data(held_out=True)\033[0m     → 25 held-out claims")
print(f"     \033[1;34m[4]\033[0m \033[1;37mget_sample_data(tiny=True, held_out=True)\033[0m → held-out subset")

# ─────────────────────────────────────────────
# QUICK LOAD CHECK
# ─────────────────────────────────────────────
try:
    _claims, _labels = get_sample_data()
    claims_ok = True
    n_claims = len(_claims)
    n_labels = len(_labels)
except Exception as e:
    claims_ok = False
    n_claims, n_labels = 0, 0
    err = str(e)

print(f"\n  \033[1;33m🔍  Load check (default queue):\033[0m")

if claims_ok:
    print(f"     \033[1;32m✅ Loaded successfully\033[0m")
    print(f"     \033[90m▸ Claims\033[0m          → \033[1;37m{n_claims}\033[0m")
    print(f"     \033[90m▸ Labeled decisions\033[0m → \033[1;37m{n_labels}\033[0m")
    matched = sum(1 for c in _claims if c["claim_id"] in _labels)
    print(f"     \033[90m▸ Matched IDs\033[0m     → \033[1;32m{matched}\033[0m / {n_claims}")
else:
    print(f"     \033[1;31m❌ Load failed:\033[0m \033[90m{err}\033[0m")
    print(f"     \033[90mCheck that {DATA} contains the expected JSON files.\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📂  CLAIMS DATASET
  JSON seed files · claims + handler decisions
════════════════════════════════════════════════════════════════════

  ⚙️   Configuration:
     ▸ Data dir        → ..\data
     ▸ Main files      → claims.json + evals/decisions.json
     ▸ Held-out files  → evals/held_out_claims.json + evals/held_out_decisions.json

  📋  Dataset variants:
     [1] get_sample_data()                  → full queue
     [2] get_sample_data(tiny=True)         → 5-claim subset
     [3] get_sample_data(held_out=True)     → 25 held-out claims
     [4] get_sample_data(tiny=True, held_out=True) → held-out subset

  🔍  Load check (default queue):
     ❌ Load failed: [Errno 2] No such file or directory: '..\\data\\evals\\decisions.json'
     Check that ..\data contains the expected JSON files.

════════════════════════════════════════════════════════════════════

